<a href="https://colab.research.google.com/github/shrutia898/Rescue911/blob/main/.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

In [ ]:
# Load the UCI Adult dataset
adult = fetch_openml(name="adult", version=2, as_frame=True)
df = adult.frame

In [ ]:
df.dtypes

,0
age,int64
workclass,category
fnlwgt,int64
education,category
education-num,int64
marital-status,category
occupation,category
relationship,category
race,category
sex,category


In [ ]:

# Drop rows with missing values for simplicity
df = df.dropna()

# Separate features X and target y
X = df.drop(columns="class")
y = df["class"]

# Convert target to binary 0/1
y = (y == '>50K').astype(int)

In [ ]:
# Split into train and test sets
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

# Further split train into retain (Dr) and forget (Df) sets
X_retain, X_forget, y_retain, y_forget = train_test_split(
    X_train_full, y_train_full, test_size=0.001, stratify=y_train_full, random_state=42)

In [ ]:
# Feature preprocessing: scale numeric, one-hot encode categorical
numeric_features = X.select_dtypes(include=['int64']).columns.tolist()
categorical_features = X.select_dtypes(include=['category', 'object']).columns.tolist()

# Create a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("Num", StandardScaler(), numeric_features),
        ("Cat", OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'  # In case some columns are neither numeric nor categorical
)

# Fit on the full training data and transform all sets
X_train_full_enc = preprocessor.fit_transform(X_train_full)
X_retain_enc = preprocessor.transform(X_retain)
X_forget_enc = preprocessor.transform(X_forget)
X_test_enc = preprocessor.transform(X_test)

# Convert to dense arrays if any part is sparse
if hasattr(X_train_full_enc, "toarray"):
    X_train_full_enc = X_train_full_enc.toarray()
    X_retain_enc = X_retain_enc.toarray()
    X_forget_enc = X_forget_enc.toarray()
    X_test_enc = X_test_enc.toarray()

# Add a column of ones for intercept
X_train_full_aug = np.hstack([X_train_full_enc, np.ones((X_train_full_enc.shape[0], 1))])
X_retain_aug = np.hstack([X_retain_enc, np.ones((X_retain_enc.shape[0], 1))])
X_forget_aug = np.hstack([X_forget_enc, np.ones((X_forget_enc.shape[0], 1))])
X_test_aug = np.hstack([X_test_enc, np.ones((X_test_enc.shape[0], 1))])

print(f"Full training set: {X_train_full_aug.shape}, Retain set Dr: {X_retain_aug.shape}, Forget set Df: {X_forget_aug.shape}, Test set: {X_test_aug.shape}")

Full training set: (36177, 97), Retain set Dr: (36140, 97), Forget set Df: (37, 97), Test set: (9045, 97)


In [ ]:
# Train logistic regression on full training data
clf_full = LogisticRegression(penalty='l2', C=1.0, fit_intercept=False,
                              solver='lbfgs', max_iter=1000, random_state=0)
clf_full.fit(X_train_full_aug, y_train_full)
w_star = clf_full.coef_.flatten()  # Parameters of model trained on Dr+Df

# Train logistic regression on retain set Dr only
clf_retrain = LogisticRegression(penalty='l2', C=1.0, fit_intercept=False,
                                 solver='lbfgs', max_iter=1000, random_state=0)
clf_retrain.fit(X_retain_aug, y_retain)
w_retrain = clf_retrain.coef_.flatten()  # Parameters of model retrained on Dr

# Train logistic regression on forget set Df only
clf_forget = LogisticRegression(penalty='l2', C=1.0, fit_intercept=False,
                                 solver='lbfgs', max_iter=1000, random_state=0)
clf_forget.fit(X_forget_aug, y_forget)
w_forget = clf_forget.coef_.flatten()  # Parameters of model retrained on Df
print(f"w* (full model) dimension: {w_star.shape[0]}")


w* (full model) dimension: 97


In [ ]:
# Define sigmoid and logistic loss functions
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def logistic_loss(X, y, w):
    """Compute average binary cross-entropy loss for dataset"""
    # Safety for extreme values
    z = X.dot(w)
    # Clip values to avoid overflow in exp
    z = np.clip(z, -20, 20)
    p = sigmoid(z)
    # Avoid log(0) by adding small epsilon
    eps = 1e-12
    loss = -(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
    return loss.mean()

def compute_gradient(X, y, w, lam=1.0):
    """Compute gradient of loss on dataset (with L2 regularization) at w"""
    m = X.shape[0]
    z = X.dot(w)
    z = np.clip(z, -20, 20)
    p = sigmoid(z)
    # Data loss gradient
    grad_data = X.T.dot(p - y) / m
    # Regularization gradient - intercept (last term) not regularized
    reg_grad = lam * w.copy()
    reg_grad[-1] = 0.0  #No regularization for intercept term
    return grad_data + reg_grad

def estimate_hessian_diag(X_f, y_f, w_star, grad_f, m=100, sigma=1e-3):
    """
    Estimate Hessian diagonal for retain data
    via small random perturbations on forget set.
    X_f, y_f: forget set features and labels
    w_star: optimal parameters w*
    grad_f: gradient on forget set at w*
    Return estimated diagonal Hessian as a vector
    """
    d = w_star.shape[0]
    base_loss = logistic_loss(X_f, y_f, w_star)
    # Initialize system Ah = b
    A = []
    b = []
    # Sample multiple small perturbations
    rng = np.random.RandomState(0)
    for i in range(m):
        δw = sigma * rng.randn(d)  #  δw ~ N(0, sigma^2 I) (sample a small random perturbation)
        w_i = w_star + δw
        # δL_f (Compute change in forget-set loss for perturbation)
        loss_i = logistic_loss(X_f, y_f, w_i)
        δL_f = loss_i - base_loss
        # 0.5 * sum_j h_j * (δw_j)^2 - grad_f · δw = δL_f  (Form equation for least squares)
        A.append(0.5 * (δw ** 2))
        b.append(δL_f + np.dot(grad_f, δw))
    A = np.vstack(A)
    b = np.array(b)
    # Hessian (Solve least-squares for h)
    h_diag, *_ = np.linalg.lstsq(A, b, rcond=None)
    # Hessian (Hessian should be positive semidefinite)
    h_diag = np.maximum(h_diag, 1e-8)

    return h_diag



In [ ]:
# Compute gradient of forget set at w*
grad_f = compute_gradient(X_forget_aug, y_forget, w_star, lam=1.0)

# Estimate diagonal Hessian for retain set using forget set and w*
hessian_diag_est = estimate_hessian_diag(
    X_forget_aug, y_forget, w_star, grad_f,
    m=100, sigma=1e-4
)

# Perform Newton-style parameter update with clipping for stability
eps = 1e-4
v_hat = grad_f / (hessian_diag_est + eps)

# Clip update step norm to avoid large parameter jumps
max_step_norm = 2.5
v_norm = np.linalg.norm(v_hat)
if v_norm > max_step_norm:
    print(f"Update step too large ({v_norm:.2f}), applying clipping")
    v_hat = v_hat * (max_step_norm / v_norm)

# Apply the unlearning update: w_unlearned = w* - H⁻¹ ∇f
w_unlearned = w_star - v_hat

# Print diagnostics
print("Hessian diagonal estimated and parameters updated with clipping.")
print(f"‖v_hat‖ (step norm): {np.linalg.norm(v_hat):.4f}")
print(f"‖w_unlearned - w_retrain‖ (distance to retrain): {np.linalg.norm(w_unlearned - w_retrain):.4f}")


Update step too large (42450.02), applying clipping
Hessian diagonal estimated and parameters updated with clipping.
‖v_hat‖ (step norm): 2.5000
‖w_unlearned - w_retrain‖ (distance to retrain): 2.5032


In [ ]:
# Define helper to compute accuracy and loss
def evaluate_model(X, y, w):
    """Return accuracy and average loss on dataset for model parameters w"""
    # Model predictions
    probs = sigmoid(np.clip(X.dot(w), -20, 20))
    preds = (probs >= 0.5).astype(int)
    acc = (preds == y).mean()
    loss = logistic_loss(X, y, w)
    return acc, loss


In [ ]:
# Compute parameter distance between unlearned and retrained models
param_dist = np.linalg.norm(w_unlearned - w_retrain)

# Evaluate performance on Dr, Df, and test sets for each model
acc_orig_dr, loss_orig_dr = evaluate_model(X_retain_aug, y_retain, w_star)
acc_retrain_dr, loss_retrain_dr = evaluate_model(X_retain_aug, y_retain, w_retrain)
acc_unlearn_dr, loss_unlearn_dr = evaluate_model(X_retain_aug, y_retain, w_unlearned)

acc_orig_df, loss_orig_df = evaluate_model(X_forget_aug, y_forget, w_star)
acc_retrain_df, loss_retrain_df = evaluate_model(X_forget_aug, y_forget, w_retrain)
acc_unlearn_df, loss_unlearn_df = evaluate_model(X_forget_aug, y_forget, w_unlearned)

acc_orig_test, loss_orig_test = evaluate_model(X_test_aug, y_test, w_star)
acc_retrain_test, loss_retrain_test = evaluate_model(X_test_aug, y_test, w_retrain)
acc_unlearn_test, loss_unlearn_test = evaluate_model(X_test_aug, y_test, w_unlearned)

# Print parameter distance and performance metrics
print(f"Parameter L2 distance between unlearned and retrained model: {param_dist:.4f}\n")

print("Accuracy and Loss on different datasets (Original vs Retrained vs Unlearned):")
print(f"Retain set (Dr) - Acc: {acc_orig_dr:.3f} vs {acc_retrain_dr:.3f} vs {acc_unlearn_dr:.3f},  "
      f"Loss: {loss_orig_dr:.4f} vs {loss_retrain_dr:.4f} vs {loss_unlearn_dr:.4f}")
print(f"Forget set (Df) - Acc: {acc_orig_df:.3f} vs {acc_retrain_df:.3f} vs {acc_unlearn_df:.3f},  "
      f"Loss: {loss_orig_df:.4f} vs {loss_retrain_df:.4f} vs {loss_unlearn_df:.4f}")
print(f"Test set        - Acc: {acc_orig_test:.3f} vs {acc_retrain_test:.3f} vs {acc_unlearn_test:.3f},  "
      f"Loss: {loss_orig_test:.4f} vs {loss_retrain_test:.4f} vs {loss_unlearn_test:.4f}")

Parameter L2 distance between unlearned and retrained model: 2.5032

Accuracy and Loss on different datasets (Original vs Retrained vs Unlearned):
Retain set (Dr) - Acc: 0.850 vs 0.850 vs 0.844,  Loss: 0.3238 vs 0.3239 vs 0.3336
Forget set (Df) - Acc: 0.811 vs 0.811 vs 0.811,  Loss: 0.4218 vs 0.4234 vs 0.3815
Test set        - Acc: 0.846 vs 0.847 vs 0.844,  Loss: 0.3248 vs 0.3248 vs 0.3357


In [ ]:
# Compute gradient of forget set at Wf
grad_forget = compute_gradient(X_forget_aug, y_forget, w_forget, lam=1.0)
# Estimate diagonal Hessian for retain set using forget set and Wf

hessian_diag_est = estimate_hessian_diag(
    X_forget_aug, y_forget, w_forget, grad_forget,
    m=100, sigma=1e-4
)

# Perform Newton-style parameter update with clipping for stability
eps = 1e-4
v_hat = grad_forget / (hessian_diag_est + eps)

# Clip update step norm to avoid large parameter jumps
max_step_norm = 2.5
v_norm = np.linalg.norm(v_hat)
if v_norm > max_step_norm:
    print(f"Update step too large ({v_norm:.2f}), applying clipping")
    v_hat = v_hat * (max_step_norm / v_norm)

# Apply the unlearning update: w_unlearned = w* - H⁻¹ ∇f
w_unlearned = w_star - v_hat

# Print diagnostics
print("Hessian diagonal estimated and parameters updated with clipping.")
print(f"‖v_hat‖ (step norm): {np.linalg.norm(v_hat):.4f}")
print(f"‖w_unlearned - w_retrain‖ (distance to retrain): {np.linalg.norm(w_unlearned - w_retrain):.4f}")

Update step too large (18104.36), applying clipping
Hessian diagonal estimated and parameters updated with clipping.
‖v_hat‖ (step norm): 2.5000
‖w_unlearned - w_retrain‖ (distance to retrain): 2.4885


In [ ]:
param_dist = np.linalg.norm(w_unlearned - w_retrain)

# Evaluate performance on Dr, Df, and test sets for each model
acc_orig_dr, loss_orig_dr = evaluate_model(X_retain_aug, y_retain, w_star)
acc_retrain_dr, loss_retrain_dr = evaluate_model(X_retain_aug, y_retain, w_retrain)
acc_unlearn_dr, loss_unlearn_dr = evaluate_model(X_retain_aug, y_retain, w_unlearned)

acc_orig_df, loss_orig_df = evaluate_model(X_forget_aug, y_forget, w_star)
acc_retrain_df, loss_retrain_df = evaluate_model(X_forget_aug, y_forget, w_retrain)
acc_unlearn_df, loss_unlearn_df = evaluate_model(X_forget_aug, y_forget, w_unlearned)

acc_orig_test, loss_orig_test = evaluate_model(X_test_aug, y_test, w_star)
acc_retrain_test, loss_retrain_test = evaluate_model(X_test_aug, y_test, w_retrain)
acc_unlearn_test, loss_unlearn_test = evaluate_model(X_test_aug, y_test, w_unlearned)

# Print parameter distance and performance metrics
print(f"Parameter L2 distance between unlearned and retrained model: {param_dist:.4f}\n")

print("Accuracy and Loss on different datasets (Original vs Retrained vs Unlearned):")
print(f"Retain set (Dr) - Acc: {acc_orig_dr:.3f} vs {acc_retrain_dr:.3f} vs {acc_unlearn_dr:.3f},  "
      f"Loss: {loss_orig_dr:.4f} vs {loss_retrain_dr:.4f} vs {loss_unlearn_dr:.4f}")
print(f"Forget set (Df) - Acc: {acc_orig_df:.3f} vs {acc_retrain_df:.3f} vs {acc_unlearn_df:.3f},  "
      f"Loss: {loss_orig_df:.4f} vs {loss_retrain_df:.4f} vs {loss_unlearn_df:.4f}")
print(f"Test set        - Acc: {acc_orig_test:.3f} vs {acc_retrain_test:.3f} vs {acc_unlearn_test:.3f},  "
      f"Loss: {loss_orig_test:.4f} vs {loss_retrain_test:.4f} vs {loss_unlearn_test:.4f}")

Parameter L2 distance between unlearned and retrained model: 2.4885

Accuracy and Loss on different datasets (Original vs Retrained vs Unlearned):
Retain set (Dr) - Acc: 0.850 vs 0.850 vs 0.774,  Loss: 0.3238 vs 0.3239 vs 0.4263
Forget set (Df) - Acc: 0.811 vs 0.811 vs 0.676,  Loss: 0.4218 vs 0.4234 vs 0.6728
Test set        - Acc: 0.846 vs 0.847 vs 0.763,  Loss: 0.3248 vs 0.3248 vs 0.4325


In [ ]:
# Original vs unlearned logits
logits_before = np.dot(X_forget_aug, w_star)
logits_after  = np.dot(X_forget_aug, w_unlearned)
delta_logits = logits_after - logits_before

# Probability difference
probs_before = sigmoid(logits_before)
probs_after  = sigmoid(logits_after)
delta_probs = np.abs(probs_after - probs_before)

# Flip rate
pred_before = (probs_before > 0.5).astype(int)
pred_after  = (probs_after > 0.5).astype(int)
flip_rate = np.mean(pred_before != pred_after)

# Output
print("Df (Forget set) behavior shift:")
print(f"Δlogits mean: {np.mean(np.abs(delta_logits)):.4f}")
print(f"Δprobs mean:  {np.mean(delta_probs):.4f}")
print(f"Prediction flip rate: {flip_rate:.4f}")

Df (Forget set) behavior shift:
Δlogits mean: 1.3269
Δprobs mean:  0.1450
Prediction flip rate: 0.2432


In [ ]:
def full_hessian_logreg(X, w, lam=1.0):
    z = X @ w
    p = sigmoid(z)
    s = p * (1 - p)  # (n,)
    Xw = X * s[:, None]
    H = Xw.T @ X/ X.shape[0] + lam * np.eye(X.shape[1])
    return H

In [ ]:
# Compute gradient of forget set at Wf
grad_forget = compute_gradient(X_forget_aug, y_forget, w_forget, lam=1.0)
# Estimate full Hessian for retain set using forget set and Wf

hessian_full = full_hessian_logreg(
    X_forget_aug, w_forget, lam=1.0
)

print(hessian_diag_est.shape)
print(grad_forget.shape)
delta = np.linalg.solve(hessian_full, grad_forget)   # (97,)
# Clip update step norm to avoid large parameter jumps
max_step_norm = 2.5
v_norm = np.linalg.norm(delta)
if v_norm > max_step_norm:
    print(f"Update step too large ({v_norm:.2f}), applying clipping")
    delta = delta * (max_step_norm / v_norm)

# Apply the unlearning update: w_unlearned = w* - H⁻¹ ∇f

w_unlearned = w_star - delta

# Print diagnostics
print("Hessian diagonal estimated and parameters updated with clipping.")
print(f"‖v_hat‖ (step norm): {np.linalg.norm(v_hat):.4f}")
print(f"‖w_unlearned - w_retrain‖ (distance to retrain): {np.linalg.norm(w_unlearned - w_retrain):.4f}")

(97,)
(97,)
Hessian diagonal estimated and parameters updated with clipping.
‖v_hat‖ (step norm): 2.5000
‖w_unlearned - w_retrain‖ (distance to retrain): 2.3571


In [ ]:
param_dist = np.linalg.norm(w_unlearned - w_retrain)

# Evaluate performance on Dr, Df, and test sets for each model
acc_orig_dr, loss_orig_dr = evaluate_model(X_retain_aug, y_retain, w_star)
acc_retrain_dr, loss_retrain_dr = evaluate_model(X_retain_aug, y_retain, w_retrain)
acc_unlearn_dr, loss_unlearn_dr = evaluate_model(X_retain_aug, y_retain, w_unlearned)

acc_orig_df, loss_orig_df = evaluate_model(X_forget_aug, y_forget, w_star)
acc_retrain_df, loss_retrain_df = evaluate_model(X_forget_aug, y_forget, w_retrain)
acc_unlearn_df, loss_unlearn_df = evaluate_model(X_forget_aug, y_forget, w_unlearned)

acc_orig_test, loss_orig_test = evaluate_model(X_test_aug, y_test, w_star)
acc_retrain_test, loss_retrain_test = evaluate_model(X_test_aug, y_test, w_retrain)
acc_unlearn_test, loss_unlearn_test = evaluate_model(X_test_aug, y_test, w_unlearned)

# Print parameter distance and performance metrics
print(f"Parameter L2 distance between unlearned and retrained model: {param_dist:.4f}\n")

print("Accuracy and Loss on different datasets (Original vs Retrained vs Unlearned):")
print(f"Retain set (Dr) - Acc: {acc_orig_dr:.3f} vs {acc_retrain_dr:.3f} vs {acc_unlearn_dr:.3f},  "
      f"Loss: {loss_orig_dr:.4f} vs {loss_retrain_dr:.4f} vs {loss_unlearn_dr:.4f}")
print(f"Forget set (Df) - Acc: {acc_orig_df:.3f} vs {acc_retrain_df:.3f} vs {acc_unlearn_df:.3f},  "
      f"Loss: {loss_orig_df:.4f} vs {loss_retrain_df:.4f} vs {loss_unlearn_df:.4f}")
print(f"Test set        - Acc: {acc_orig_test:.3f} vs {acc_retrain_test:.3f} vs {acc_unlearn_test:.3f},  "
      f"Loss: {loss_orig_test:.4f} vs {loss_retrain_test:.4f} vs {loss_unlearn_test:.4f}")

Parameter L2 distance between unlearned and retrained model: 2.3571

Accuracy and Loss on different datasets (Original vs Retrained vs Unlearned):
Retain set (Dr) - Acc: 0.850 vs 0.850 vs 0.800,  Loss: 0.3238 vs 0.3239 vs 0.4077
Forget set (Df) - Acc: 0.811 vs 0.811 vs 0.649,  Loss: 0.4218 vs 0.4234 vs 0.7107
Test set        - Acc: 0.846 vs 0.847 vs 0.797,  Loss: 0.3248 vs 0.3248 vs 0.4122
